In [1]:
from pathlib import Path

dataset_path = Path("../data")
dataset_filename = "competencia_01_crudo.csv"

file_path = dataset_path / dataset_filename

print(file_path)

..\data\competencia_01_crudo.csv


In [2]:
print(file_path.exists())

True


In [3]:
import duckdb
import pandas as pd

In [4]:
%load_ext sql

In [5]:
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

%sql duckdb:///

In [6]:
%sql duckdb:///

In [7]:
%%sql
CREATE OR REPLACE TABLE competencia_01_crudo AS
SELECT *
FROM read_csv_auto('../data/competencia_01_crudo.csv');

,Success


In [8]:
%sql SELECT * FROM competencia_01_crudo LIMIT 5

,numero_de_cliente,foto_mes,active_quarter,cliente_vip,internet,cliente_edad,cliente_antiguedad,mrentabilidad,mrentabilidad_annual,mcomisiones,...,Visa_madelantodolares,Visa_fultimo_cierre,Visa_mpagado,Visa_mpagospesos,Visa_mpagosdolares,Visa_fechaalta,Visa_mconsumototal,Visa_cconsumos,Visa_cadelantosefectivo,Visa_mpagominimo
0,12159854,202103,1,0,0,56,134,688.41,26701.84,98.82,...,0.0,1,0.0,-16247.77,0.0,4056,15732.34,1,0,1137.81
1,12159858,202103,1,0,0,48,102,78.43,24418.75,-73.62,...,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,NaN
2,12160484,202103,1,0,0,60,55,8101.55,3162.23,13399.50,...,0.0,1,0.0,-31103.23,0.0,1632,2860.54,2,0,19858.89
3,12160591,202103,1,0,0,46,275,14825.78,138050.05,1146.27,...,0.0,1,0.0,-13733.44,0.0,2122,1419.36,3,0,1231.65
4,12160747,202103,1,0,0,47,194,2015.61,31240.49,1791.25,...,0.0,1,0.0,0.00,0.0,5901,1286.93,1,0,82.11


In [9]:
%%sql

SELECT
    foto_mes,
    COUNT(*) AS cantidad
FROM competencia_01_crudo
GROUP BY foto_mes
ORDER BY foto_mes;

,foto_mes,cantidad
0,202103,162900
1,202104,163284
2,202105,163768
3,202106,164114
4,202107,164348
5,202108,164647


In [10]:
%%sql

SELECT
    numero_de_cliente,
    foto_mes,
    mrentabilidad
FROM competencia_01_crudo
WHERE numero_de_cliente = 12159854
ORDER BY foto_mes;

,numero_de_cliente,foto_mes,mrentabilidad
0,12159854,202103,688.41
1,12159854,202104,1622.57
2,12159854,202105,1987.98
3,12159854,202106,2580.75
4,12159854,202107,2686.85
5,12159854,202108,2707.62


In [11]:
%%sql

SELECT
    numero_de_cliente,
    foto_mes,
    mrentabilidad,
    LEAD(mrentabilidad, 1) OVER (
        PARTITION BY numero_de_cliente
        ORDER BY foto_mes
    ) AS mrentabilidad_mas_1,
    LAG(mrentabilidad, 1) OVER (
        PARTITION BY numero_de_cliente
        ORDER BY foto_mes
    ) AS mrentabilidad_menos_1,
    LEAD(mrentabilidad, 2) OVER (
        PARTITION BY numero_de_cliente
        ORDER BY foto_mes
    ) AS mrentabilidad_mas_2,
    LAG(mrentabilidad, 2) OVER (
        PARTITION BY numero_de_cliente
        ORDER BY foto_mes
    ) AS mrentabilidad_menos_2
FROM competencia_01_crudo
WHERE numero_de_cliente = 12159854
ORDER BY foto_mes;

,numero_de_cliente,foto_mes,mrentabilidad,mrentabilidad_mas_1,mrentabilidad_menos_1,mrentabilidad_mas_2,mrentabilidad_menos_2
0,12159854,202103,688.41,1622.57,NaN,1987.98,NaN
1,12159854,202104,1622.57,1987.98,688.41,2580.75,NaN
2,12159854,202105,1987.98,2580.75,1622.57,2686.85,688.41
3,12159854,202106,2580.75,2686.85,1987.98,2707.62,1622.57
4,12159854,202107,2686.85,2707.62,2580.75,NaN,1987.98
5,12159854,202108,2707.62,NaN,2686.85,NaN,2580.75


In [12]:
%%sql

SELECT
    numero_de_cliente,
    foto_mes
FROM competencia_01_crudo
WHERE numero_de_cliente = 60112254
ORDER BY foto_mes;

,numero_de_cliente,foto_mes
0,60112254,202103
1,60112254,202105
2,60112254,202106
3,60112254,202107
4,60112254,202108


In [13]:
%%sql

WITH periodos AS (
    SELECT DISTINCT foto_mes
    FROM competencia_01_crudo
),
clientes AS (
    SELECT DISTINCT numero_de_cliente
    FROM competencia_01_crudo
)
SELECT
    numero_de_cliente,
    foto_mes
FROM clientes
CROSS JOIN periodos
WHERE numero_de_cliente = 60112254
ORDER BY foto_mes;

,numero_de_cliente,foto_mes
0,60112254,202103
1,60112254,202104
2,60112254,202105
3,60112254,202106
4,60112254,202107
5,60112254,202108


In [14]:
%%sql

WITH periodos AS (
    SELECT DISTINCT foto_mes
    FROM competencia_01_crudo
),
clientes AS (
    SELECT DISTINCT numero_de_cliente
    FROM competencia_01_crudo
),
todo AS (
    SELECT
        numero_de_cliente,
        foto_mes
    FROM clientes
    CROSS JOIN periodos
)
SELECT
    t.numero_de_cliente,
    t.foto_mes,
    IF(c.numero_de_cliente IS NULL, 0, 1) AS mes_0
FROM todo t
LEFT JOIN competencia_01_crudo c
    USING (numero_de_cliente, foto_mes)
WHERE t.numero_de_cliente = 60112254
ORDER BY foto_mes;

,numero_de_cliente,foto_mes,mes_0
0,60112254,202103,1
1,60112254,202104,0
2,60112254,202105,1
3,60112254,202106,1
4,60112254,202107,1
5,60112254,202108,1


In [15]:
%%sql

CREATE OR REPLACE TABLE competencia_01 AS

WITH periodos AS (
    SELECT DISTINCT foto_mes
    FROM competencia_01_crudo
),
clientes AS (
    SELECT DISTINCT numero_de_cliente
    FROM competencia_01_crudo
),
todo AS (
    SELECT
        numero_de_cliente,
        foto_mes
    FROM clientes
    CROSS JOIN periodos
),
clase_ternaria AS (
    SELECT
        c.*,
        IF(c.numero_de_cliente IS NULL, 0, 1) AS mes_0,
        LEAD(mes_0, 1) OVER (
            PARTITION BY t.numero_de_cliente
            ORDER BY foto_mes
        ) AS mes_1,
        LEAD(mes_0, 2) OVER (
            PARTITION BY t.numero_de_cliente
            ORDER BY foto_mes
        ) AS mes_2,
        CASE
            WHEN mes_1 = 1 AND mes_2 = 1 THEN 'CONTINUA'
            WHEN mes_1 = 1 AND mes_2 = 0 THEN 'BAJA+2'
            WHEN mes_1 = 0 AND mes_2 = 0 THEN 'BAJA+1'
            ELSE NULL
        END AS clase_ternaria
    FROM todo t
    LEFT JOIN competencia_01_crudo c
        USING (numero_de_cliente, foto_mes)
)
SELECT
    * EXCLUDE (mes_0, mes_1, mes_2)
FROM clase_ternaria
WHERE mes_0 = 1;

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Success


In [16]:
%%sql

SELECT
    numero_de_cliente,
    foto_mes,
    clase_ternaria
FROM competencia_01
WHERE numero_de_cliente = 60112254
ORDER BY foto_mes;

,numero_de_cliente,foto_mes,clase_ternaria
0,60112254,202103,NaN
1,60112254,202105,CONTINUA
2,60112254,202106,CONTINUA
3,60112254,202107,NaN
4,60112254,202108,NaN


In [17]:
%sql SELECT COUNT(*) AS cantidad FROM competencia_01

,cantidad
0,983061


In [18]:
%%sql

PIVOT competencia_01
ON clase_ternaria
USING COUNT(numero_de_cliente)
GROUP BY foto_mes
ORDER BY foto_mes;

,foto_mes,BAJA+1,BAJA+2,CONTINUA
0,202103,1018,960,160921
1,202104,957,1139,161181
2,202105,1139,870,161755
3,202106,871,1098,162142
4,202107,0,0,0
5,202108,0,0,0


In [19]:
%%sql

PIVOT (
    SELECT
        foto_mes,
        numero_de_cliente,
        CASE
            WHEN clase_ternaria IS NULL THEN 'SIN_TARGET'
            ELSE clase_ternaria
        END AS clase_ternaria
    FROM competencia_01
)
ON clase_ternaria
USING COUNT(numero_de_cliente)
GROUP BY foto_mes
ORDER BY foto_mes;

,foto_mes,BAJA+1,BAJA+2,CONTINUA,SIN_TARGET
0,202103,1018,960,160921,1
1,202104,957,1139,161181,7
2,202105,1139,870,161755,4
3,202106,871,1098,162142,3
4,202107,0,0,0,164348
5,202108,0,0,0,164647


In [20]:
%%sql

COPY competencia_01
TO '../data/competencia_01.csv'
(FORMAT CSV, HEADER);

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Success
